In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import glob
import shutil
from scipy.stats import pearsonr

In [ ]:
colabfold_path = "/home/biocomp/Documents/Jazmin/localcolabfold/.pixi/envs/default/bin/colabfold_batch"
base_dir = os.getcwd()
alphafold_dir = os.path.join(base_dir, "results_alphaFold")

In [ ]:
def matrix_comparison(target_protein, sequence):
    prediction = os.path.join(alphafold_dir, f"{sequence}.csv")
    real = os.path.join(base_dir, "real_distances", f"{target_protein}.csv")
    
    if not os.path.exists(real):
        raise FileNotFoundError(f"Real matrix not found: {real}")
    if not os.path.exists(prediction):
        raise FileNotFoundError(f"Predicted matrix not found: {prediction}")

    real_matrix = pd.read_csv(real).to_numpy()
    prediction_matrix = pd.read_csv(prediction).to_numpy()

    if prediction_matrix.shape != real_matrix.shape:
        raise ValueError("Matrices must have the same dimensions.")

    indices_triup = np.triu_indices_from(prediction_matrix, k=1)
    v1 = prediction_matrix[indices_triup]
    v2 = real_matrix[indices_triup]

    drmsd = np.sqrt(np.mean((v1 - v2)**2))
    corr, _ = pearsonr(v1, v2)

    return {
        "dRMSD": drmsd,
        "Pearson_Correlation": corr
    }

In [ ]:
def get_representative_coords_from_pdb(pdb_path):
    residues = {}
    residue_order = []

    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM  "):
                atom_name = line[12:16].strip()
                res_name = line[17:20].strip()
                chain_id = line[21]
                res_seq = line[22:27].strip()
                
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())

                res_key = f"{chain_id}_{res_seq}"

                if res_key not in residues:
                    residues[res_key] = {'name': res_name, 'CA': None, 'CB': None}
                    residue_order.append(res_key)

                if atom_name == 'CA':
                    residues[res_key]['CA'] = np.array([x, y, z])
                elif atom_name == 'CB':
                    residues[res_key]['CB'] = np.array([x, y, z])

    coords = []
    for key in residue_order:
        res = residues[key]
        if res['name'] == 'GLY':
            coord = res['CA']
        else:
            coord = res['CB'] if res['CB'] is not None else res['CA']
        
        if coord is not None:
            coords.append(coord)
            
    return np.array(coords)

In [ ]:
def calculate_gdt_ts(target_protein, sequence):
    # 1. Define paths
    target_pdb_path = os.path.join(base_dir, "initialize", "pdb_files", f"{target_protein}.pdb")
    pred_coords_path = os.path.join(alphafold_dir, f"{sequence}_coords.npy")

    # 2. Load coordinates
    if not os.path.exists(target_pdb_path):
        raise FileNotFoundError(f"Target PDB not found: {target_pdb_path}")
    if not os.path.exists(pred_coords_path):
        raise FileNotFoundError(f"Predicted coordinates file not found: {pred_coords_path}")

    coords_target = get_representative_coords_from_pdb(target_pdb_path)
    coords_pred = np.load(pred_coords_path)

    # 3. Match lengths (just in case)
    min_len = min(len(coords_target), len(coords_pred))
    coords_target = coords_target[:min_len]
    coords_pred = coords_pred[:min_len]

    # 4. Kabsch algorithm (structural superposition)
    centroid_target = np.mean(coords_target, axis=0)
    centroid_pred = np.mean(coords_pred, axis=0)
    
    c_target = coords_target - centroid_target
    c_pred = coords_pred - centroid_pred

    H = c_pred.T @ c_target
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T #matricial product

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    coords_pred_rotated = (c_pred @ R.T) + centroid_target

    # 5. Distance calculation and GDT_TS
    distances = np.linalg.norm(coords_target - coords_pred_rotated, axis=1)
    n = len(distances)

    p1 = np.sum(distances <= 1.0) / n
    p2 = np.sum(distances <= 2.0) / n
    p4 = np.sum(distances <= 4.0) / n
    p8 = np.sum(distances <= 8.0) / n

    return (p1 + p2 + p4 + p8) / 4.0

In [ ]:
def get_alphafold_files(sequence):
    os.makedirs(alphafold_dir, exist_ok=True)
    dist_matrix_file = os.path.join(alphafold_dir, f"{sequence}.csv")
    
    coords_file = os.path.join(alphafold_dir, f"{sequence}_coords.npy")

    if not os.path.exists(dist_matrix_file) or not os.path.exists(coords_file):
        fasta_path = os.path.join(base_dir, f"{sequence}.fasta")
        with open(fasta_path, "w") as f:
            f.write(f">{sequence}\n{sequence}")
            
        custom_env = os.environ.copy()
        custom_env["MPLBACKEND"] = "Agg"
        custom_env["TF_CPP_MIN_LOG_LEVEL"] = "0"
        custom_env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
        custom_env["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"
        
        custom_env["TF_FORCE_UNIFIED_MEMORY"] = "1"
        custom_env["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "4.0"
        
        try:
            result = subprocess.run([
                colabfold_path, fasta_path, alphafold_dir,
                "--num-recycle", "3"
            ], check=True, env=custom_env, capture_output=True, text=True)

            pdb_files = glob.glob(os.path.join(alphafold_dir, f"{sequence}*rank_001*.pdb"))
            if not pdb_files:
                raise FileNotFoundError(f"AlphaFold did not generate a PDB for {sequence}")

            coords_pred = get_representative_coords_from_pdb(pdb_files[0])
            
            np.save(coords_file, coords_pred) 

            # 2. Calculate matrix distances
            N = len(coords_pred)
            matrix_pred = np.zeros((N, N))
            for i in range(N):
                for j in range(N):
                    matrix_pred[i, j] = np.linalg.norm(coords_pred[i] - coords_pred[j])

            column_names = [f"c{i+1}" for i in range(N)]
            df_matrix = pd.DataFrame(matrix_pred, columns=column_names)
            df_matrix.to_csv(dist_matrix_file, index=False)

        except subprocess.CalledProcessError as e:
            print(f"ColabFold error detected. Code: {e.returncode}")
            raise
        finally:
            
            for item in glob.glob(os.path.join(alphafold_dir, f"{sequence}*")):
              
                if not item.endswith(f"{sequence}.csv") and not item.endswith(f"{sequence}_coords.npy"):
                    if os.path.isdir(item):
                        shutil.rmtree(item)
                    else:
                        os.remove(item)
            
            
            for extra in ["cite.bibtex", "config.json", "log.txt"]:
                f_extra = os.path.join(alphafold_dir, extra)
                if os.path.exists(f_extra):
                    os.remove(f_extra)
            if os.path.exists(fasta_path):
                os.remove(fasta_path)

In [ ]:
#target = "1llm"
#sequence = "MEKWYCCACCGPIHTEDDIAKARMHWARN" 
#get_alphafold_files(sequence)

# 1s7m
# 3sb1
# 1llm

#resultados = matrix_comparation(target, sequence)
#print(f"dRMSD: {resultados['dRMSD']:.4f} Å")
#print(f"Correlación: {resultados['Pearson_Correlation']:.4f}")

In [ ]:
#resultado = calculate_gdt_ts("1llm", "MEKWYCCACCGPIHTEDDIAKARMHWARN")
#print(f"GDT_TS Score: {resultado['GDT_TS']:.4f}")

#target = "1llm"
#sequence = "MEKWYCCACCGPIHTEDDIAKARMHWARN" 
#get_alphafold_files(sequence)

## Running all results

In [ ]:
target_proteins = [
    "1aay", "1emn", "1f2i", "1llm", "1lql", "1msl", "1n7d", "1np6", "1o6w", "1p9n",
    "1r5l", "1s7m", "2e45", "2i13", "2kxq", "2lkm", "2oar", "2prt", "2qiw", "2qq8",
    "3c8v", "3clq", "3e7r", "3ewk", "3h25", "3m9q", "3sb1", "3vdu", "3w68", "4gzn", "5uiy"
]

In [ ]:
blosum62_fitness_auc_pr = os.path.join(base_dir, "results_esm2/blosum62_fitness_auc_pr")
blosum62_fitness_shannon_entropy = os.path.join(base_dir, "results_esm2/blosum62_fitness_shannon_entropy")
metrics_results_dir = os.path.join(alphafold_dir, "metrics_results")

In [ ]:
import time
import os
import pandas as pd

def evaluate_proteins(csv_path, target_protein, name_csv, list_indiv = [0, 14, 29, 44, 59, 74, 89]):
    start_time = time.time()
    
    # File paths
    file_name = f"genetic_results_esm2_{target_protein}.csv"
    full_path = os.path.join(csv_path, file_name)
    
    if not os.path.exists(full_path):
        print(f"Error: File not found {full_path}")
        return

    df = pd.read_csv(full_path)
    indices = list_indiv
    #indices = [0, 9, 19, 29, 39, 49, 59, 69, 79, 89, 99]
    metric_results = []

    for idx in indices:
        if idx >= len(df):
            continue
            
        sequence = df.iloc[idx]['sequence']
        print(f"Evaluating sequence {idx+1} for {target_protein}...")
        
        try:
            # Generate files (PDB and .npy coordinates)
            get_alphafold_files(sequence)
            
            # Matrix comparison (dRMSD, Pearson)
            res_matrix = matrix_comparison(target_protein, sequence)
            
            # Compute GDT_TS (new structural metric)
            gdt_score = calculate_gdt_ts(target_protein, sequence)
            
            metric_results.append({
                "sequence": sequence,
                "dRMSD": res_matrix["dRMSD"],
                "Pearson_Correlation": res_matrix["Pearson_Correlation"],
                "GDT_TS": gdt_score  # <-- New column added
            })
            
        except Exception as e:
            print(f"Error processing row {idx+1}: {e}")

    # Save results
    if metric_results:
        total_time = time.time() - start_time
        df_final = pd.DataFrame(metric_results)
        
        # Save metrics CSV
        path_csv_out = os.path.join(metrics_results_dir, f"{name_csv}_{target_protein}.csv")
        df_final.to_csv(path_csv_out, index=False)
        
        # Save runtime log
        path_txt_out = os.path.join(alphafold_dir, f"{name_csv}_{target_protein}_running_time.txt")
        with open(path_txt_out, "w") as f:
            f.write(f"Protein: {target_protein}\nTotal time: {total_time:.2f}s\n")
            
        print(f"--- Finished: {target_protein} ---")
        print(f"CSV saved at: {metrics_results_dir}")

In [ ]:
'''
for i, proteina in enumerate(target_proteins, 1):
    print(f"\n[{i}/{len(target_proteins)}] >>> Processing: {proteina}")
    
    evaluate_proteins(blosum62_fitness_auc_pr, proteina, "results_blosum62_fitness_auc_pr")
'''

In [ ]:
'''
for i, proteina in enumerate(target_proteins, 1):
    print(f"\n[{i}/{len(target_proteins)}] >>> Processing: {proteina}")
    
    evaluate_proteins(blosum62_fitness_shannon_entropy, proteina, "results_blosum62_fitness_shannon_entropy")
'''

## Random sequences

In [ ]:
import os
import time
import random
import pandas as pd

def get_target_length(target_protein):
    target_pdb_path = os.path.join(base_dir, "initialize", "pdb_files", f"{target_protein}.pdb")
    if not os.path.exists(target_pdb_path):
        raise FileNotFoundError(f"Target PDB not found: {target_pdb_path}")
    
    coords = get_representative_coords_from_pdb(target_pdb_path)
    return len(coords)

In [ ]:
def generate_random_sequence(length):
    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    return "".join(random.choice(amino_acids) for _ in range(length))

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import zlib

def random_proteins(target_protein, num_sequences=7, seed=26):

    if seed is not None:
        peptide_hash = zlib.crc32(target_protein.encode('utf-8'))
        unique_seed = seed + (peptide_hash % 1000000)
        
        random.seed(unique_seed)
        np.random.seed(unique_seed)
        print(f"Assigned seed for {target_protein}: {unique_seed}")

    start_time = time.time()
    
    random_results_dir = os.path.join(alphafold_dir, "results_random")
    os.makedirs(random_results_dir, exist_ok=True)
    
    try:
        target_len = get_target_length(target_protein)
    except Exception as e:
        print(f"Error retrieving target length for {target_protein}: {e}")
        return
        
    print(f"\nGenerating {num_sequences} random sequences of length {target_len} for {target_protein}...")
    
    metric_results = []
    
    for idx in range(num_sequences):
        sequence = generate_random_sequence(target_len)
        print(f"Evaluating random sequence {idx+1}/{num_sequences}...")
        
        try:
            get_alphafold_files(sequence)
            
            res_matrix = matrix_comparison(target_protein, sequence)
            
            gdt_score = calculate_gdt_ts(target_protein, sequence)
            
            metric_results.append({
                "sequence": sequence,
                "dRMSD": res_matrix["dRMSD"],
                "Pearson_Correlation": res_matrix["Pearson_Correlation"],
                "GDT_TS": gdt_score
            })
            
        except Exception as e:
            print(f"Error processing random sequence {idx+1}: {e}")

    if metric_results:
        total_time = time.time() - start_time
        df_final = pd.DataFrame(metric_results)
        
        path_csv_out = os.path.join(random_results_dir, f"results_random_{target_protein}.csv")
        df_final.to_csv(path_csv_out, index=False)
        
        path_txt_out = os.path.join(random_results_dir, f"results_random_{target_protein}_running_time.txt")
        with open(path_txt_out, "w") as f:
            f.write(f"Protein: {target_protein}\nTotal time: {total_time:.2f}s\n")
            
        print(f"--- Completed: {target_protein} (Random) ---")
        print(f"CSV saved in: {random_results_dir}")

In [ ]:
for i, protein in enumerate(target_proteins, 1):
    print(f"\n[{i}/{len(target_proteins)}] >>> Processing Random Baseline: {protein}")
    random_proteins(protein, num_sequences=7)


[1/31] >>> Processing Random Baseline: 1aay

Generating 7 random sequences of length 26 for 1aay...
Evaluating random sequence 1/7...
Evaluating random sequence 2/7...
Evaluating random sequence 3/7...
